In [81]:
import openpyxl
import numpy as np

def write_matrix_to_excel(matrix, filename="predicts.xlsx", sheet_name="Sheet1"):
    try:
        # 尝试打开已有的工作簿
        wb = openpyxl.load_workbook(filename)
    except FileNotFoundError:
        # 如果文件不存在，创建一个新的工作簿
        wb = openpyxl.Workbook()
        # 删除默认创建的工作表
    if sheet_name in wb.sheetnames:
        del wb[sheet_name]

    if isinstance(matrix, np.ndarray):
        matrix = matrix.tolist()

    # 创建一个新的工作表
    ws = wb.create_sheet(sheet_name)

    # 将矩阵写入Excel
    for row_idx, row in enumerate(matrix, start=1):
        for col_idx, value in enumerate(row, start=1):
            ws.cell(row=row_idx, column=col_idx, value=value)

    # 保存Excel文件
    wb.save(filename)
    print(f"矩阵已成功写入 {filename} 的工作表 {sheet_name}")

In [82]:
import torch
import torch.nn.functional as F

def predict(model, net_dict, data, label):
    """
    此函数用于判断模型的分类是否准确。

    参数:
    model (torch.nn.Module): PyTorch 网络模型。
    net_dict (dict): 模型的参数。
    data (torch.Tensor): 输入的数据。
    label (torch.Tensor): 数据对应的标签。

    返回:
    int: 1 表示模型分类准确，0 表示模型分类错误。
    """
    model.load_state_dict(net_dict)
    model.eval()
    with torch.no_grad():
        out = model(data)

    probabilities = F.softmax(out, dim=1)
    predictions = torch.argmax(probabilities, dim=1)
    correct = (predictions == label)
    return predictions


In [83]:
subid = 9

In [84]:
from scipy import io
import os
import torch

from Model.DLSSNet.model_zhengjiao import Net as DLSSNet
from Model.DLSSNet.args_zjBCIsingletrain_val import data_config as DLSS_config

from Model.EEGNet.EEGNet_baseline import EEGNet_baseline as EEGNet
from Model.EEGNet.args_BCI_trainval import data_config as eeg_config

from Model.ConFormer_forAnalysis.Conformer import Conformer
from Model.ConFormer_forAnalysis.arg_BCI import data_config as Conformer_config

from Model.baseline_FBCNet.networks import deepConvNet
from Model.baseline_FBCNet.arg_BCI import data_config as deepConvNet_config

from Model.baseLine_MATT.mAtt import mAtt_bci as matt

from torch.backends import cudnn
from einops import rearrange

device = torch.device('cpu')
cudnn.benchmark = False
cudnn.deterministic = True


data_path = 'C:/2023Experiment/DATA/BCICIV_2a_mat'
file_e = io.loadmat(os.path.join(data_path, 'BCIC_S' + f'{subid:02d}' + '_E.mat'))
x_e = torch.Tensor(file_e['x_test'])
y_e = torch.Tensor(file_e['y_test']).view(-1)
results = []

x_all = x_e
y_all = y_e
x_all = x_all[:, :, 124:562].to(device)
y_all = y_all.long().to(device)
labels = y_all.tolist()
results.append(labels)


In [85]:

# for DLSSNet
DLSSmodel = DLSSNet(num_channels=DLSS_config.num_channel, 
                    len_window=DLSS_config.len_window, 
                    d_model=DLSS_config.d_model, 
                    frame_stride=DLSS_config.frame_stride,
                    num_frame=DLSS_config.num_frame,
                    num_head=DLSS_config.num_head, 
                    encoder_num_layers=DLSS_config.encoder_num_layers, 
                    low_p = DLSS_config.low_p,
                    dropout=DLSS_config.dropout, 
                    transformerparwiseforward_dimrat=DLSS_config.transformerparwiseforward_dimrat, 
                    statenum=DLSS_config.statenum,
                    num_class=DLSS_config.num_class)


path = 'C:\\2023Experiment\\parametersensitive\\单被试训练测试_matt数据_d_60\\BCICom_2a\\Conv+TransFormer+GPoolingV2+ClassTransHead+decoder+正交约束\\ON2024-11-22At13-10-37\\ckpl\\Fold'+str(subid).zfill(2)
cpkl_path = os.path.join(path, 'Conv+TransFormer+GPoolingV2+ClassTransHead+decoder+正交约束_best_params.pkl')
net_dict = torch.load(cpkl_path, map_location=device)
net_dict = net_dict['net_state_dict']
result = predict(DLSSmodel, net_dict, x_all, y_all)
result=result.numpy().astype(int).tolist()
results.append(result)



In [86]:
path = 'C:/2023Experiment/TrainingForanalysis/单被试训练测试conformer无数据扩增/ckpl/Fold'+str(subid).zfill(2)
cpkl_path = os.path.join(path, 'Conformer_best_params.pkl')
eegConformer = Conformer()
net_dict = torch.load(cpkl_path, map_location=device)
net_dict = net_dict['net_state_dict']
result = predict(eegConformer, net_dict, x_all, y_all)
result=result.numpy().astype(int).tolist()
results.append(result)

In [87]:
path = 'C:/2023Experiment/TrainingForanalysis/EEGNet训练测试/ckpl/Fold'+str(subid).zfill(2)
cpkl_path = os.path.join(path, 'BaseLine-EEGNet_best_params.pkl')
EEGnetmodel = EEGNet(**eeg_config.params)
net_dict = torch.load(cpkl_path, map_location=device)
net_dict = net_dict['net_state_dict']
result = predict(EEGnetmodel, net_dict, x_all, y_all)
result=result.numpy().astype(int).tolist()
results.append(result)

In [88]:
path = 'C:/2023Experiment/TrainingForanalysis/deepconv训练结果/ckpl/Fold'+str(subid).zfill(2)
cpkl_path = os.path.join(path, 'DeepConv_best_params.pkl')
deepConvNetmodel = deepConvNet(nChan=deepConvNet_config.num_channel, nClass=deepConvNet_config.num_class, nTime=deepConvNet_config.timepoint)
net_dict = torch.load(cpkl_path, map_location=device)
net_dict = net_dict['net_state_dict']
result = predict(deepConvNetmodel, net_dict, x_all, y_all)
result=result.numpy().astype(int).tolist()
results.append(result)

In [89]:
path = 'C:/2023Experiment/TrainingForanalysis/matt训练结果/ckpl/Fold'+str(subid).zfill(2)
cpkl_path = os.path.join(path, 'mATT_baseline_best_params.pkl')
mattmodel = matt(3)
net_dict = torch.load(cpkl_path, map_location=device)
net_dict = net_dict['net_state_dict']
result = predict(mattmodel, net_dict, x_all, y_all)
result=result.numpy().astype(int).tolist()
results.append(result)

In [90]:
write_matrix_to_excel(results, filename="predicts.xlsx", sheet_name=str(subid).zfill(2))

矩阵已成功写入 predicts.xlsx 的工作表 09
